# PCA Analysis on Breast Cancer Dataset
## Milestone 2 - Anderson Cancer Center

This notebook applies PCA to the breast cancer dataset from sklearn to identify the most essential variables. The dataset is reduced from 30 features down to 2 principal components.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Load and Explore the Dataset

In [ ]:
# Load the breast cancer dataset
cancer_dataset = load_breast_cancer()
X = cancer_dataset.data
y = cancer_dataset.target
feature_names = cancer_dataset.feature_names
target_names = cancer_dataset.target_names

# Put data into a DataFrame
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y
df['target_name'] = df['target'].map({0: target_names[0], 1: target_names[1]})

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print(f"\nSamples: {df.shape[0]}, Features: {len(feature_names)}")
print(f"Target classes: {target_names}")
print("\nClass distribution:")
print(df['target_name'].value_counts())

## 3. Standardize the Data

PCA is sensitive to feature scale, so we need to standardize before applying it.

In [ ]:
# Check for missing values
print("Missing values:", df[feature_names].isnull().sum().sum())

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nScaled data shape: {X_scaled.shape}")
print(f"Mean (first 5 features, should be ~0): {X_scaled.mean(axis=0)[:5].round(4)}")
print(f"Std (first 5 features, should be ~1):  {X_scaled.std(axis=0)[:5].round(4)}")

## 4. Apply PCA - Reduce to 2 Components

In [ ]:
# Apply PCA with 2 components
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Store results in a DataFrame
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['target'] = y
pca_df['target_name'] = pca_df['target'].map({0: target_names[0], 1: target_names[1]})

print(f"Original shape: {X_scaled.shape}")
print(f"Reduced shape:  {X_pca.shape}")
print(f"Dimensionality reduction: {((1 - X_pca.shape[1]/X_scaled.shape[1])*100):.1f}%")

## 5. Visualize the PCA Results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for target, target_name in enumerate(target_names):
    indices = pca_df['target'] == target
    ax.scatter(pca_df.loc[indices, 'PC1'],
               pca_df.loc[indices, 'PC2'],
               label=target_name,
               alpha=0.7,
               s=100,
               edgecolors='black',
               linewidth=0.5)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}% variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}% variance)', fontsize=12)
ax.set_title('PCA - Cancer Dataset Projected to 2 Components', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Variance Explained and Feature Contributions

This section shows how much variance each component captures and which original features contribute the most.

In [ ]:
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print(f"PC1 explains: {explained_variance[0]*100:.2f}%")
print(f"PC2 explains: {explained_variance[1]*100:.2f}%")
print(f"Total variance captured: {cumulative_variance[1]*100:.2f}%")

# Feature loadings - how much each original feature contributes
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
loading_df = pd.DataFrame(loadings, columns=['PC1', 'PC2'], index=feature_names)

print("\nTop 5 features contributing to PC1:")
pc1_top = loading_df['PC1'].abs().sort_values(ascending=False).head(5)
for feature in pc1_top.index:
    print(f"  {feature}: {loading_df.loc[feature, 'PC1']:.4f}")

print("\nTop 5 features contributing to PC2:")
pc2_top = loading_df['PC2'].abs().sort_values(ascending=False).head(5)
for feature in pc2_top.index:
    print(f"  {feature}: {loading_df.loc[feature, 'PC2']:.4f}")

# Plot feature loadings
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pc1_sorted = loading_df['PC1'].sort_values()
axes[0].barh(range(len(pc1_sorted)), pc1_sorted.values, color='steelblue')
axes[0].set_yticks(range(len(pc1_sorted)))
axes[0].set_yticklabels([name.replace('_', ' ') for name in pc1_sorted.index], fontsize=8)
axes[0].set_xlabel('Loading Value')
axes[0].set_title('Feature Loadings for PC1', fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

pc2_sorted = loading_df['PC2'].sort_values()
axes[1].barh(range(len(pc2_sorted)), pc2_sorted.values, color='coral')
axes[1].set_yticks(range(len(pc2_sorted)))
axes[1].set_yticklabels([name.replace('_', ' ') for name in pc2_sorted.index], fontsize=8)
axes[1].set_xlabel('Loading Value')
axes[1].set_title('Feature Loadings for PC2', fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Cumulative variance plot
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 'bo-', linewidth=2, markersize=8)
plt.axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(range(1, 3))
plt.tight_layout()
plt.show()

## 7. Bonus: Logistic Regression on PCA Components

Using the 2 PCA components to train a logistic regression classifier and see how well it performs with the reduced features.

In [ ]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.3, random_state=42, stratify=y
)

# Train logistic regression
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train, y_train)

y_train_pred = log_reg.predict(X_train)
y_test_pred = log_reg.predict(X_test)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

## 8. Model Evaluation

In [ ]:
# Compute metrics
train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_recall    = recall_score(y_train, y_train_pred)
train_f1        = f1_score(y_train, y_train_pred)

test_accuracy  = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall    = recall_score(y_test, y_test_pred)
test_f1        = f1_score(y_test, y_test_pred)

print("Training Set:")
print(f"  Accuracy:  {train_accuracy:.4f}")
print(f"  Precision: {train_precision:.4f}")
print(f"  Recall:    {train_recall:.4f}")
print(f"  F1-Score:  {train_f1:.4f}")

print("\nTest Set:")
print(f"  Accuracy:  {test_accuracy:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  F1-Score:  {test_f1:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
print("\nConfusion Matrix (Test Set):")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=target_names))

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix - Logistic Regression (PCA Features)', fontweight='bold')
plt.tight_layout()
plt.show()

# Compare train vs test metrics
metrics_names  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
train_metrics  = [train_accuracy, train_precision, train_recall, train_f1]
test_metrics   = [test_accuracy, test_precision, test_recall, test_f1]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(metrics_names))
width = 0.35

ax.bar(x - width/2, train_metrics, width, label='Train', alpha=0.8)
ax.bar(x + width/2, test_metrics,  width, label='Test',  alpha=0.8)

ax.set_ylabel('Score')
ax.set_title('Train vs Test Metrics', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.set_ylim([0.85, 1.0])
ax.grid(axis='y', alpha=0.3)

for i, (tv, sv) in enumerate(zip(train_metrics, test_metrics)):
    ax.text(i - width/2, tv + 0.004, f'{tv:.3f}', ha='center', fontsize=9)
    ax.text(i + width/2, sv + 0.004, f'{sv:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Summary

- PCA reduced the cancer dataset from 30 features to just 2 principal components, capturing about 63% of the total variance.
- The scatter plot shows decent separation between malignant and benign cases in the 2D PCA space.
- Features like worst radius, worst perimeter, and worst area had the highest loadings on PC1, meaning they contribute the most to that component.
- The logistic regression model trained on only 2 PCA components still achieved around 95% accuracy on the test set, which shows that PCA preserved the important information needed for classification.
- This kind of dimensionality reduction is useful for identifying which variables are most important, which can help when presenting findings to stakeholders or making a case for donor funding.